# P05: calibrate the cost model on a Colab GPU (Phase 2.3)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all** (about 10 minutes,
most of it installing the NVIDIA HPC SDK).

It measures what the cost model needs (memory bandwidth, arithmetic rate, OpenMP
startup and thread-team cost on the host; kernel launch, host↔device copies, bandwidth,
arithmetic rate and device startup on the GPU) and prints a JSON machine file. **Copy
the JSON the last cell prints back to Claude**; it becomes `machine/colab-t4.json`, used
as `p05tool -machine=machine/colab-t4.json`.

Generated by `eval/calibrate/make_notebook.py` from `eval/calibrate/calibrate.{c,py}`:
edit those, not this notebook.


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%writefile /content/calibrate.c
// Microbenchmarks for the cost model's machine parameters (Phase 2.3).
// calibrate.py builds this, runs each benchmark (the startup ones in fresh
// processes) and writes machine/<name>.json; see README.md.
//
//   calibrate <bench> [args]      prints one number on stdout
//
// Host:   stream1 / streamN   triad bandwidth, 1 thread / all, GB/s
//         ops1 / opsN         ops/s as p05tool counts them, 1 thread / all, G/s
//         first / first_if0   this process's first `parallel` (if(0)), us
//         region              an empty `parallel for` once the runtime is up, us
// Device (built with -DP05_TARGET):
//         tfirst              this process's first `target`, us
//         tlaunch             an empty kernel once the device is up, us
//         tcopy <bytes> to|from   an empty `target data` mapping that many
//                             bytes (allocate, copy one way, free), us
//         tstream / tops      triad bandwidth / ops on the device, G.../s
//
// Bytes and ops are counted the way the cost model counts them
// (Profitability.cpp): the triad touches 24 B per element (no write-allocate
// traffic); the ops kernel is 64 multiply-adds written out, 128 ops with no
// loop counter among them (the model counts a k++ too, but it costs next to
// nothing, so a counted loop would inflate the rate), which
// `p05tool eval/calibrate/calibrate.c` reports as its loop's ops/iteration.
#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#define FMA(x) x = x * 0.999 + 0.001;
#define FMA8(x) FMA(x) FMA(x) FMA(x) FMA(x) FMA(x) FMA(x) FMA(x) FMA(x)
#define FMA64(x) FMA8(x) FMA8(x) FMA8(x) FMA8(x) FMA8(x) FMA8(x) FMA8(x) FMA8(x)
#define OPS_PER_ELEM 128

static double now_us(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return ts.tv_sec * 1e6 + ts.tv_nsec / 1e3;
}

static double *alloc(size_t n, double v) {
  double *p = malloc(n * sizeof(double));
  if (!p) {
    fprintf(stderr, "out of memory\n");
    exit(1);
  }
  for (size_t i = 0; i < n; i++)
    p[i] = v + (double)(i % 7);
  return p;
}

// The kernels the cost model would see: p05tool counts 128 ops per
// iteration of `ops_kernel`'s loop.
void triad(double *a, const double *b, const double *c, long n) {
  for (long i = 0; i < n; i++)
    a[i] = b[i] + 0.5 * c[i];
}

void ops_kernel(double *out, const double *in, long n) {
  for (long i = 0; i < n; i++) {
    double x = in[i];
    FMA64(x)
    out[i] = x;
  }
}

static void triad_par(double *a, const double *b, const double *c, long n) {
#pragma omp parallel for schedule(static)
  for (long i = 0; i < n; i++)
    a[i] = b[i] + 0.5 * c[i];
}

static void ops_par(double *out, const double *in, long n) {
#pragma omp parallel for schedule(static)
  for (long i = 0; i < n; i++) {
    double x = in[i];
    FMA64(x)
    out[i] = x;
  }
}

// The arguments of one benchmarked call.
struct args {
  double *a, *b, *c;
  long n;
};

// Best of `reps` runs of f, in us.
static double best(int reps, void (*f)(struct args *), struct args *x) {
  double b = 1e300;
  for (int r = 0; r < reps; r++) {
    double t0 = now_us();
    f(x);
    double t = now_us() - t0;
    if (t < b)
      b = t;
  }
  return b;
}

static void run_triad(struct args *x) { triad(x->a, x->b, x->c, x->n); }
static void run_triad_par(struct args *x) { triad_par(x->a, x->b, x->c, x->n); }
static void run_ops(struct args *x) { ops_kernel(x->a, x->b, x->n); }
static void run_ops_par(struct args *x) { ops_par(x->a, x->b, x->n); }

static volatile double sink;

static double host_stream(int threads) {
  const long n = 1L << 25;
  struct args x = {alloc(n, 0), alloc(n, 1), alloc(n, 2), n};
  double us;
  if (threads == 1)
    us = best(10, run_triad, &x);
  else {
    run_triad_par(&x); // thread start
    us = best(10, run_triad_par, &x);
  }
  sink = x.a[n / 3];
  return 24.0 * n / us / 1e3; // GB/s
}

static double host_ops(int threads) {
  const long n = 1L << 20;
  struct args x = {alloc(n, 0), alloc(n, 1), NULL, n};
  double us;
  if (threads == 1)
    us = best(5, run_ops, &x);
  else {
    run_ops_par(&x);
    us = best(5, run_ops_par, &x);
  }
  sink = x.a[n / 3];
  return (double)OPS_PER_ELEM * n / us / 1e3; // G ops/s
}

// A run-time false, so the compiler can't drop the construct.
static volatile int zero = 0;

static double first_region(int if0) {
  int x = 0;
  const int on = zero;
  double t0 = now_us();
  if (if0) {
#pragma omp parallel if (on)
    x++;
  } else {
#pragma omp parallel
    {
#pragma omp atomic
      x++;
    }
  }
  double t = now_us() - t0;
  sink = x;
  return t;
}

static double empty_region(void) {
  const int nt = omp_get_max_threads();
  int dummy[256] = {0};
  for (int w = 0; w < 100; w++) {
#pragma omp parallel for
    for (int i = 0; i < nt; i++)
      dummy[i % 256] += i;
  }
  const int reps = 2000;
  double t0 = now_us();
  for (int r = 0; r < reps; r++) {
#pragma omp parallel for
    for (int i = 0; i < nt; i++)
      dummy[i % 256] += i;
  }
  sink = dummy[0];
  return (now_us() - t0) / reps;
}

#ifdef P05_TARGET
static double target_first(void) {
  int on_host = 1;
  double t0 = now_us();
#pragma omp target map(from : on_host)
  { on_host = omp_is_initial_device(); }
  double t = now_us() - t0;
  if (on_host) {
    fprintf(stderr, "target region ran on the host\n");
    exit(1);
  }
  return t;
}

static double target_launch(void) {
  int x = 0;
  for (int w = 0; w < 20; w++) {
#pragma omp target teams distribute parallel for
    for (int i = 0; i < 1; i++)
      x += 0;
  }
  const int reps = 1000;
  double t0 = now_us();
  for (int r = 0; r < reps; r++) {
#pragma omp target teams distribute parallel for
    for (int i = 0; i < 1; i++)
      x += 0;
  }
  return (now_us() - t0) / reps;
}

static void map_to(struct args *x) {
  double *a = x->a;
  const long n = x->n;
#pragma omp target data map(to : a[0:n])
  {}
}

static void map_from(struct args *x) {
  double *a = x->a;
  const long n = x->n;
#pragma omp target data map(from : a[0:n])
  {}
}

static double target_copy(long bytes, int to) {
  struct args x = {alloc(bytes / (long)sizeof(double), 1), NULL, NULL,
                   bytes / (long)sizeof(double)};
  target_launch(); // device up
  const int reps = bytes >= (1L << 26) ? 5 : 20;
  double us = best(reps, to ? map_to : map_from, &x);
  sink = x.a[0];
  return us;
}

static void dev_triad(struct args *x) {
  double *a = x->a, *b = x->b, *c = x->c;
  const long n = x->n;
#pragma omp target teams distribute parallel for
  for (long i = 0; i < n; i++)
    a[i] = b[i] + 0.5 * c[i];
}

static void dev_ops(struct args *x) {
  double *out = x->a, *in = x->b;
  const long n = x->n;
#pragma omp target teams distribute parallel for
  for (long i = 0; i < n; i++) {
    double v = in[i];
    FMA64(v)
    out[i] = v;
  }
}

// Data stays on the device across the timed kernels.
static double target_stream(void) {
  const long n = 1L << 25;
  struct args x = {alloc(n, 0), alloc(n, 1), alloc(n, 2), n};
  double us;
#pragma omp target data map(to : x.b[0:n], x.c[0:n]) map(from : x.a[0:n])
  {
    dev_triad(&x);
    us = best(10, dev_triad, &x);
  }
  sink = x.a[n / 3];
  return 24.0 * n / us / 1e3;
}

static double target_ops(void) {
  const long n = 1L << 24;
  struct args x = {alloc(n, 0), alloc(n, 1), NULL, n};
  double us;
#pragma omp target data map(to : x.b[0:n]) map(from : x.a[0:n])
  {
    dev_ops(&x);
    us = best(5, dev_ops, &x);
  }
  sink = x.a[n / 3];
  return (double)OPS_PER_ELEM * n / us / 1e3;
}
#endif

int main(int argc, char **argv) {
  if (argc < 2) {
    fprintf(stderr, "usage: calibrate <bench> [args]\n");
    return 2;
  }
  const char *b = argv[1];
  double v;
  if (!strcmp(b, "stream1"))
    v = host_stream(1);
  else if (!strcmp(b, "streamN"))
    v = host_stream(0);
  else if (!strcmp(b, "ops1"))
    v = host_ops(1);
  else if (!strcmp(b, "opsN"))
    v = host_ops(0);
  else if (!strcmp(b, "first"))
    v = first_region(0);
  else if (!strcmp(b, "first_if0"))
    v = first_region(1);
  else if (!strcmp(b, "region"))
    v = empty_region();
  else if (!strcmp(b, "threads"))
    v = omp_get_max_threads();
#ifdef P05_TARGET
  else if (!strcmp(b, "tfirst"))
    v = target_first();
  else if (!strcmp(b, "tlaunch"))
    v = target_launch();
  else if (!strcmp(b, "tcopy") && argc == 4)
    v = target_copy(atol(argv[2]), !strcmp(argv[3], "to"));
  else if (!strcmp(b, "tstream"))
    v = target_stream();
  else if (!strcmp(b, "tops"))
    v = target_ops();
#endif
  else {
    fprintf(stderr, "unknown benchmark %s\n", b);
    return 2;
  }
  printf("%.6f\n", v);
  return 0;
}


In [ ]:
%%writefile /content/calibrate.py
#!/usr/bin/env python3
"""Measure the cost model's machine parameters and write a -machine file.

    eval/calibrate/calibrate.py --name m4 --out machine/m4.json      (host)
    eval/calibrate/calibrate.py --name colab-t4 --gpu --cc nvc \\
        --cflags "-O2 -mp=gpu -gpu=cc75" --out machine/colab-t4.json

Builds calibrate.c (with -DP05_TARGET for --gpu), runs each benchmark and
derives the fields README.md lists. Startup costs are the median of 7 fresh
processes minus the steady-state cost of the same construct; everything else
is the best of several runs inside one process. Raw numbers go under "raw".
"""
import argparse
import datetime
import json
import os
import platform
import shlex
import statistics
import subprocess
import sys
import tempfile

HERE = os.path.dirname(os.path.abspath(__file__))
FRESH = 7  # fresh processes per startup measurement


def run(binary, *args, env=None):
    r = subprocess.run([binary, *map(str, args)], capture_output=True, text=True,
                       env={**os.environ, **(env or {})})
    if r.returncode != 0:
        sys.exit(f"{os.path.basename(binary)} {' '.join(map(str, args))} failed "
                 f"({r.returncode}): {r.stderr.strip()}")
    return float(r.stdout.strip())


def fresh(binary, *args, env=None):
    return [run(binary, *args, env=env) for _ in range(FRESH)]


def fit_copy(points):
    """Least-squares t = latency + bytes / bandwidth over the sizes of at
    least 1 MiB (smaller ones are all latency); returns (us, GB/s)."""
    pts = [(b, t) for b, t in points if b >= 1 << 20]
    n = len(pts)
    mb = sum(b for b, _ in pts) / n
    mt = sum(t for _, t in pts) / n
    slope = (sum((b - mb) * (t - mt) for b, t in pts) /
             sum((b - mb) ** 2 for b, _ in pts))  # us per byte
    return max(mt - slope * mb, 0.0), 1.0 / slope / 1e3


def main():
    ap = argparse.ArgumentParser(description=__doc__,
                                 formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--name", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--cc", default=os.environ.get("CC", "clang"))
    ap.add_argument("--cflags", default="-O2 -fopenmp")
    ap.add_argument("--gpu", action="store_true", help="also measure the device")
    ap.add_argument("--notes", default="")
    a = ap.parse_args()

    work = tempfile.mkdtemp(prefix="p05cal")
    binary = os.path.join(work, "calibrate")
    cmd = [a.cc, *shlex.split(a.cflags), *(["-DP05_TARGET"] if a.gpu else []),
           os.path.join(HERE, "calibrate.c"), "-o", binary]
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True)
    version = next((l.strip() for l in subprocess.run(
        [a.cc, "--version"], capture_output=True, text=True).stdout.splitlines()
                    if l.strip()), a.cc)  # nvc prints a blank line first

    raw = {}

    def step(key, f):
        print(f"  {key} ...", end=" ", flush=True)
        raw[key] = f()
        print(raw[key], flush=True)
        return raw[key]

    threads = step("threads", lambda: run(binary, "threads"))
    stream1 = step("stream_1thread_gbs", lambda: run(binary, "stream1"))
    streamn = step("stream_all_gbs", lambda: run(binary, "streamN"))
    ops1 = step("ops_1thread_gops", lambda: run(binary, "ops1"))
    opsn = step("ops_all_gops", lambda: run(binary, "opsN"))
    region = step("empty_parallel_for_us", lambda: run(binary, "region"))
    first = step("first_parallel_us", lambda: fresh(binary, "first"))
    first0 = step("first_parallel_if0_us", lambda: fresh(binary, "first_if0"))

    cpu = {
        "throughput_gflops": round(ops1, 3),
        # The model's aggregate is throughput * cores: cores is how many of
        # one thread all the threads together are worth (unequal cores and
        # static schedules make it fewer than the hardware count).
        "cores": max(1, round(opsn / ops1)),
        "dram_bandwidth_gbs": round(streamn, 3),
        "single_core_dram_share": round(min(stream1 / streamn, 1.0), 3),
        "thread_start_us": round(region, 3),
        "runtime_init_us": round(max(statistics.median(first) - region, 0.0), 3),
    }
    out = {
        "name": a.name,
        "measured": datetime.date.today().isoformat(),
        "host": f"{platform.machine()} {platform.processor() or platform.system()}, "
                f"{int(threads)} OpenMP threads",
        "compiler": f"{version} ({a.cflags})",
        "cpu": cpu,
    }
    if a.gpu:
        env = {"OMP_TARGET_OFFLOAD": "MANDATORY"}
        launch = step("empty_kernel_us", lambda: run(binary, "tlaunch", env=env))
        tfirst = step("first_target_us", lambda: fresh(binary, "tfirst", env=env))
        sizes = [1 << k for k in range(10, 29, 2)]
        to = step("map_to_us", lambda: [[b, run(binary, "tcopy", b, "to", env=env)]
                                         for b in sizes])
        frm = step("map_from_us", lambda: [[b, run(binary, "tcopy", b, "from", env=env)]
                                           for b in sizes])
        lat_to, bw_to = fit_copy(to)
        lat_from, bw_from = fit_copy(frm)
        raw["map_to_fit"] = {"latency_us": lat_to, "gbs": bw_to}
        raw["map_from_fit"] = {"latency_us": lat_from, "gbs": bw_from}
        tstream = step("device_stream_gbs", lambda: run(binary, "tstream", env=env))
        tops = step("device_ops_gops", lambda: run(binary, "tops", env=env))
        out["gpu"] = {
            "kernel_launch_us": round(launch, 3),
            # One bandwidth prices both directions: the per-byte average.
            "pcie_bandwidth_gbs": round(2.0 / (1.0 / bw_to + 1.0 / bw_from), 3),
            "throughput_gflops": round(tops, 3),
            "bandwidth_gbs": round(tstream, 3),
            "init_us": round(max(statistics.median(tfirst) - launch, 0.0), 3),
        }
        smi = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version",
                              "--format=csv,noheader"], capture_output=True, text=True)
        out["device"] = smi.stdout.strip() or "unknown"
    out["notes"] = a.notes or (
        "Ops are counted as p05tool counts them (calibrate.c). Startup costs: "
        f"median of {FRESH} fresh processes minus the steady-state construct.")
    out["raw"] = raw
    os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
    with open(a.out, "w") as f:
        json.dump(out, f, indent=2)
        f.write("\n")
    print(json.dumps({k: out[k] for k in ("name", "cpu", "gpu") if k in out}, indent=2))
    print("wrote", a.out)


if __name__ == "__main__":
    main()


In [ ]:
%%bash
cd /content
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
python3 calibrate.py --name colab-t4 --gpu --cc "$NVC" --cflags "-O2 -mp=gpu -gpu=cc$CC" \
  --notes "Colab $(nvidia-smi --query-gpu=name --format=csv,noheader | head -1), host $(nproc) vCPU; nvc -mp=gpu" \
  --out colab-t4.json
echo; echo "===== copy everything below this line back to Claude ====="
cat colab-t4.json
